# NB01 — Identificación del TCLab con el kit real: ¿hay evidencia de orden fraccionario?

**Proyecto:** fPINN + sintonía diferenciable de FOPID (Kp, Ki, Kd, λ, μ) en el TCLab — artículo objetivo: *Control Engineering Practice*.

**Versión solo kit.** Este notebook adquiere los datos con el TCLab físico (sin emulador) y luego los analiza.
La prueba de control con planta simulada de orden conocido quedó en la versión anterior (`NB01_identificacion_TCLab.ipynb`).

**Objetivo (hipótesis H1):** decidir si el TCLab muestra un comportamiento fraccionario que justifique la línea,
comparando modelos de orden entero y fraccionario con los mismos datos.

| Modelo | Parámetros | Papel |
|---|---|---|
| FOPDT dos puntos (35,3 % / 85,3 %) | K, τ, θ | Referencia de la literatura (Oliveira, Hedengren y Rossiter, IFAC 2020) |
| FOPDT mínimos cuadrados | K, τ, θ | Línea base clásica |
| SOPDT mínimos cuadrados | K, τ1, τ2, θ | **Competidor fuerte**: explica la forma en S con dos constantes enteras |
| FFOPDT (Grünwald-Letnikov) | K, τ_α, α, θ | Candidato fraccionario: τ_α·D^α y + y = K·u(t−θ) |
| Modelo físico 2×2 (APMonitor) | U, Us, α1, α2, τs | Caja gris: balance de energía con convección, radiación, acoplamiento y sensor |

**Experimentos con el kit**
- **A** — escalones en Q1 (Q2 = 0): 0→35→55→15 %, 900 s cada uno (≈ 45 min) → canales G11 (Q1→T1) y G21 (Q1→T2).
- **B** — escalones en Q2 (Q1 = 0): igual que A (≈ 45 min) → G12 (Q2→T1, camino de la perturbación) y G22 (Q2→T2).
- **C** — APRBS en Q1 (Q2 = 0): niveles aleatorios 15–55 %, permanencias 20–300 s, 60 min, semilla 7. Validación exigente (sección 10).

**Niveles (6 nov):** la primera corrida con 0→50→80 % se abortó a los 1200 s porque T1 llegó a 85 °C. Ajuste FOPDT
sobre esos datos (`expA_cortado`): K ≈ 0,77 °C/%, τ ≈ 185 s, θ ≈ 25 s (IFAC 2020: 0,76 / 127–152 s / 20–29 s).
Con 50 % T1 llegó a 66,3 °C a los 900 s; con 80 % habría subido a ≈ 89 °C. Con 55 % como máximo, el estado
estacionario queda en ≈ T_AMB + 42 °C ≈ 70–72 °C con ambiente de 28–30 °C. Antes de calentar, el notebook
verifica que T_AMB + K_EST·(nivel máximo) no supere `T_PLAN_MAX`; si no cabe, se detiene sin calentar.

Antes de cada experimento el notebook **espera el equilibrio térmico** (T1 y T2 estables y cerca del ambiente).
Entre experimentos el kit tarda 20–40 min en enfriarse, así que la sesión completa dura **≈ 3 a 4 h**.

**Antes de correr**
1. Kit conectado por USB **y** fuente de potencia en el conector de los calentadores (nunca en el jack del Arduino).
2. Cerrar NB00 u otro programa que use el puerto (Arduino IDE, otro notebook): solo una conexión a la vez.
3. Kit lejos de ventiladores, aire acondicionado y sol directo; no tocar los calentadores durante la corrida.
4. Correr las celdas en orden. Para detener: botón ■ de Jupyter; los calentadores se apagan solos.
   Si el kernel se cuelga o muere, correr la celda **"Apagado de emergencia"** o desconectar el USB.

**Reanudar:** los experimentos ya completos (CSV en `NB01_out/`) se saltan; basta volver a correr el notebook.
Para repetir uno, borrar su CSV o poner `OVERWRITE = True`. Con `ACQUIRE = False` solo se reanalizan los CSV.

In [ ]:
import os, json, time, datetime, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import tclab

# --- Conexión y adquisición -------------------------------------------------
PORT = 'COM3'              # puerto del Arduino; None = detección automática
ACQUIRE = True             # False = no usar el kit, solo reanalizar los CSV de NB01_out/
RUN_EXP_A, RUN_EXP_B, RUN_EXP_C = True, True, True
OVERWRITE = False          # True = repetir experimentos aunque su CSV ya esté completo

# --- Equilibrio térmico antes de cada experimento ----------------------------
T_AMB = None               # temperatura ambiente [°C]; None = se mide con el kit frío en la sección 1
EQ_WINDOW = 120            # ventana para juzgar estabilidad [s]
EQ_TOL = 0.2               # variación máxima de T1 y T2 en la ventana [°C]
EQ_AMB_TOL = 1.5           # |Ti − T_AMB| máximo para considerar "en ambiente" [°C]
EQ_T12_TOL = 1.0           # |T1 − T2| máximo [°C]
EQ_MAX_WAIT = 3600         # espera máxima [s]; si se agota, se avisa y el experimento arranca igual
T_MAX = 85.0               # seguridad: si T1 o T2 superan este valor se aborta y se apaga todo [°C]
K_EST = 0.80               # ganancia Q1->T1 con margen [°C/%] (medida 6 nov: 0,77)
T_PLAN_MAX = 78.0          # chequeo previo: T_AMB + K_EST·nivel máx. debe quedar por debajo [°C]

FAST = False               # True = menos multiarranques en los ajustes (solo prueba rápida)
OUT_DIR = 'NB01_out'
os.makedirs(OUT_DIR, exist_ok=True)

TS = 1.0                                       # periodo de muestreo [s]
STEP_PLAN = [(0, 0), (10, 35), (910, 55), (1810, 15)]   # (t_inicio [s], nivel [%])
T_END = 2710                                   # duración de A y B [s]
ID_WINDOW = (0, 910)                           # primer escalón -> identificación
VAL_WINDOW = (910, T_END)                      # escalones 55 % y 15 % -> validación
SEGMENTS = [(10, 910), (910, 1810), (1810, 2710)]  # tramos para modelos locales

CSV_A = os.path.join(OUT_DIR, 'expA.csv')      # Q1 en escalones, Q2 = 0
CSV_B = os.path.join(OUT_DIR, 'expB.csv')      # Q2 en escalones, Q1 = 0
META = os.path.join(OUT_DIR, 'metadatos_kit.json')
N_STARTS = 2 if FAST else 5

# Experimento C: APRBS en Q1 (Q2 = 0). La semilla fija la señal.
CSV_C = os.path.join(OUT_DIR, 'expC.csv')
C_T_END = 3600                 # 60 min
C_HOLD = (20, 300)             # permanencia mínima y máxima en cada nivel [s] (máx ≈ 1,6τ)
C_LEVELS = (15, 55)            # rango de Q1 [%] (máx. igual al de A/B)
C_MIN_JUMP = 15                # salto mínimo entre niveles consecutivos [%]
C_SEED = 7

QUANT = 0.322                  # cuantización del TMP36 con ADC de 10 bits [°C]

# Referencia publicada (IFAC 2020, lazo Q1->T1): para comparar el kit propio
REF_IFAC = {'2pt': dict(K=0.76, tau=127, theta=28.7), 'PSO': dict(K=0.78, tau=152, theta=19.7)}

def step_profile(t, plan=STEP_PLAN):
    """Nivel del calentador [%] en el tiempo t según el plan de escalones."""
    level = 0.0
    for t0, q in plan:
        if t >= t0:
            level = q
    return level

def aprbs_plan(t_end=None, hold=None, levels=None, min_jump=None, t0=10, seed=None):
    """APRBS: niveles y tiempos de permanencia aleatorios. Devuelve [(t_inicio, nivel)] como STEP_PLAN."""
    t_end = C_T_END if t_end is None else t_end; hold = C_HOLD if hold is None else hold
    levels = C_LEVELS if levels is None else levels; min_jump = C_MIN_JUMP if min_jump is None else min_jump
    rng = np.random.default_rng(C_SEED if seed is None else seed)
    plan, t, last = [(0, 0.0)], t0, 0.0
    while t < t_end:
        q = rng.uniform(*levels)
        while abs(q - last) < min_jump:
            q = rng.uniform(*levels)
        plan.append((int(t), round(float(q), 1))); last = q
        t += int(round(rng.uniform(*hold)))
    return plan

C_PLAN = aprbs_plan()
pd.DataFrame(C_PLAN, columns=['t_inicio_s', 'Q1_pct']).to_csv(os.path.join(OUT_DIR, 'plan_expC.csv'), index=False)

meta = json.load(open(META)) if os.path.exists(META) else {}
def save_meta():
    with open(META, 'w') as f:
        json.dump(meta, f, indent=2, default=str)
print(f'tclab {tclab.__version__} | Python {platform.python_version()} | {platform.platform()}')
print(f'Plan C: {len(C_PLAN) - 1} niveles en {C_T_END} s')

## 1. Conexión con el kit y prueba rápida

Abre la conexión, muestra el firmware, lee T1 y T2 durante 10 s con los calentadores apagados y parpadea el LED.
Si `T_AMB = None`, la temperatura ambiente de referencia se toma aquí, **con el kit frío** (al inicio de la sesión,
antes de calentar). Si el kit ya estaba caliente, poner `T_AMB` a mano con un termómetro de la sala.

In [ ]:
def open_lab():
    """Conecta con el TCLab. Si quedó una conexión abierta de una corrida anterior, la cierra primero."""
    global lab_handle
    try:
        lab_handle.close()
    except Exception:
        pass
    lab_handle = tclab.TCLab(port=PORT) if PORT else tclab.TCLab()
    lab_handle.Q1(0); lab_handle.Q2(0)
    return lab_handle

def heaters_off(lab):
    for f in (lambda: lab.Q1(0), lambda: lab.Q2(0), lambda: lab.LED(0)):
        try: f()
        except Exception as e: print('  aviso al apagar:', e)

lab_handle = None
if ACQUIRE:
    lab = open_lab()
    try:
        meta.update(fecha=datetime.datetime.now().isoformat(timespec='seconds'), puerto=getattr(lab, 'port', PORT),
                    firmware=getattr(lab, 'version', '?'), tclab=tclab.__version__, so=platform.platform(),
                    python=platform.python_version(), procesador=platform.processor())
        lab.LED(100)
        r = []
        for t in tclab.clock(10, 1):
            r.append((lab.T1, lab.T2))
            lab.LED(100 if int(t) % 2 == 0 else 0)
        r = np.array(r)
        print(f'T1 = {r[:, 0].mean():.2f} ± {r[:, 0].std():.2f} °C | T2 = {r[:, 1].mean():.2f} ± {r[:, 1].std():.2f} °C')
        if T_AMB is None and 'T_AMB' not in meta:
            if abs(r[:, 0].mean() - r[:, 1].mean()) < EQ_T12_TOL:
                meta['T_AMB'] = float(r.mean()); meta['T_AMB_origen'] = 'medida con el kit frío'
            else:
                print('Aviso: T1 y T2 difieren; el kit no parece frío. Poner T_AMB a mano.')
        elif T_AMB is not None:
            meta['T_AMB'] = float(T_AMB); meta['T_AMB_origen'] = 'fijada a mano'
        print('T_AMB de referencia:', meta.get('T_AMB'))
        save_meta()
    finally:
        heaters_off(lab)

## 2. Adquisición con el kit

Para cada experimento:
1. **Chequeo de equilibrio**: T1 y T2 varían menos de `EQ_TOL` en `EQ_WINDOW` s, |T1 − T2| < `EQ_T12_TOL` y ambas
   están a menos de `EQ_AMB_TOL` de `T_AMB`. Si se agota `EQ_MAX_WAIT`, se avisa, se registra en los metadatos y el
   experimento arranca igual (el análisis usa variables de desviación respecto al estado inicial).
2. Se lee T1 y T2, luego se aplica Q (convención del análisis). Cada fila se escribe y se guarda en el acto en
   `expX.part.csv`; al terminar se renombra a `expX.csv`. Si se interrumpe, el `.part.csv` queda para revisar.
3. Ante cualquier error, interrupción (■) o temperatura mayor que `T_MAX`, los calentadores se apagan (`try/finally`).

Se registran también el tiempo real de cada muestra (`t_real`) y el retraso del reloj, para documentar la calidad
del muestreo en el artículo.

In [ ]:
def wait_equilibrium(lab, label):
    t_amb = meta.get('T_AMB')
    print(f'[{label}] esperando equilibrio (T_AMB = {t_amb})')
    t0 = time.time(); hist = []
    for _ in tclab.clock(EQ_MAX_WAIT, 5):
        now, T1, T2 = time.time(), lab.T1, lab.T2
        hist.append((now, T1, T2))
        if max(T1, T2) > T_MAX:
            raise RuntimeError(f'Temperatura {max(T1, T2):.1f} °C > T_MAX con calentadores apagados: revisar el kit')
        old = [h for h in hist if h[0] <= now - EQ_WINDOW]
        if not old:
            continue
        dT = max(abs(T1 - old[-1][1]), abs(T2 - old[-1][2]))
        d12 = abs(T1 - T2)
        damb = max(abs(T1 - t_amb), abs(T2 - t_amb)) if t_amb is not None else 0.0
        print(f'\r  {now - t0:5.0f} s  T1={T1:.2f}  T2={T2:.2f}  ΔT({EQ_WINDOW} s)={dT:.2f}  '
              f'|T1−T2|={d12:.2f}  |T−Tamb|={damb:.2f}   ', end='')
        if dT < EQ_TOL and d12 < EQ_T12_TOL and damb < EQ_AMB_TOL:
            print('\n  equilibrio OK'); return dict(ok=True, espera_s=now - t0, T1_0=T1, T2_0=T2)
    print('\n  AVISO: no se alcanzó el equilibrio en EQ_MAX_WAIT; el experimento arranca igual')
    return dict(ok=False, espera_s=time.time() - t0, T1_0=hist[-1][1], T2_0=hist[-1][2])

def csv_complete(fname, t_end):
    if not os.path.exists(fname): return False
    try: return pd.read_csv(fname)['time'].max() >= t_end - 2 * TS
    except Exception: return False

def run_kit_experiment(lab, label, heater, fname, plan, t_end):
    if csv_complete(fname, t_end) and not OVERWRITE:
        print(f'[{label}] ya completo ({fname}); se salta'); return
    part = fname.replace('.csv', '.part.csv')
    q_max = max(q for _, q in plan); t_amb = meta.get('T_AMB') or 30.0
    if t_amb + K_EST * q_max > T_PLAN_MAX:
        raise RuntimeError(f'[{label}] plan inseguro: {t_amb:.1f} + {K_EST}·{q_max} = {t_amb + K_EST * q_max:.1f} °C '
                           f'> T_PLAN_MAX = {T_PLAN_MAX} °C. Bajar los niveles (no se calentó).')
    if os.path.exists(part):                       # corrida anterior cortada: se guarda con otro nombre
        old_part = part.replace('.part.csv', f'_cortado_{datetime.datetime.now():%Y%m%d_%H%M%S}.csv')
        os.replace(part, old_part); print(f'[{label}] corrida cortada anterior guardada como {old_part}')
    try:
        heaters_off(lab)
        eq = wait_equilibrium(lab, label)
        print(f'[{label}] adquiriendo {t_end} s con el calentador {heater} -> {fname}')
        lags, t_wall0 = [], time.time()
        with open(part, 'w') as f:
            f.write('time,Q1,Q2,T1,T2,t_real\n')
            for t in tclab.clock(t_end - TS, TS):
                T1, T2 = lab.T1, lab.T2           # leer primero, luego actuar
                if max(T1, T2) > T_MAX:
                    raise RuntimeError(f'T = {max(T1, T2):.1f} °C > T_MAX = {T_MAX} °C: experimento abortado')
                q = step_profile(t, plan)
                q1, q2 = (q, 0.0) if heater == 1 else (0.0, q)
                lab.Q1(q1); lab.Q2(q2)
                tr = time.time() - t_wall0; lags.append(tr - t)
                f.write(f'{t:.2f},{q1},{q2},{T1:.3f},{T2:.3f},{tr:.3f}\n'); f.flush()
                if int(t) % 60 == 0:
                    print(f'\r  t={t:5.0f}/{t_end} s  Q{heater}={q:5.1f} %  T1={T1:6.2f}  T2={T2:6.2f}   ', end='')
        os.replace(part, fname)
        lags = np.array(lags)
        meta[f'exp{label}'] = dict(**eq, inicio=datetime.datetime.fromtimestamp(t_wall0).isoformat(timespec='seconds'),
                                   duracion_s=float(time.time() - t_wall0), muestras=len(lags),
                                   retraso_reloj_medio_s=float(lags.mean()), retraso_reloj_max_s=float(lags.max()),
                                   T_max_medida=float(pd.read_csv(fname)[['T1', 'T2']].values.max()))
        save_meta()
        print(f'\n[{label}] listo: {len(lags)} muestras, retraso máx. del reloj {lags.max():.2f} s')
    finally:
        heaters_off(lab)

if ACQUIRE:
    lab = open_lab()
    try:
        if RUN_EXP_A: run_kit_experiment(lab, 'A', 1, CSV_A, STEP_PLAN, T_END)
        if RUN_EXP_B: run_kit_experiment(lab, 'B', 2, CSV_B, STEP_PLAN, T_END)
        if RUN_EXP_C: run_kit_experiment(lab, 'C', 1, CSV_C, C_PLAN, C_T_END)
    finally:
        heaters_off(lab)
        try: lab.close()
        except Exception: pass

### Apagado de emergencia

Correr esta celda si la adquisición se cortó de forma anormal (kernel colgado, error de conexión) y hay dudas de que
los calentadores quedaron apagados. Si no responde, desconectar el cable USB y la fuente.

In [ ]:
try:
    heaters_off(lab_handle); lab_handle.close()
except Exception:
    try:
        _lab = tclab.TCLab(port=PORT) if PORT else tclab.TCLab(); _lab.Q1(0); _lab.Q2(0); _lab.close()
    except Exception as e:
        print('No se pudo conectar para apagar:', e, '-> desconectar USB y fuente')
print('Calentadores apagados.')

## 3. Carga de los datos

Los CSV se llevan a una rejilla exacta de 1 s (interpolación lineal en el tiempo medido), por si `tclab.clock`
se atrasó en alguna muestra. El análisis usa los experimentos A y B; el C es opcional (sección 9).

In [ ]:
def load_csv(fname, t_end):
    raw = pd.read_csv(fname)
    t_raw = raw['time'].values - raw['time'].values[0]
    t = np.arange(0, t_end, TS)
    df = pd.DataFrame({'time': t})
    for c in ['T1', 'T2']:
        df[c] = np.interp(t, t_raw, raw[c].values)
    for c in ['Q1', 'Q2']:            # entradas: retener el valor (no interpolar escalones)
        idx = np.clip(np.searchsorted(t_raw, t, side='right') - 1, 0, len(raw) - 1)
        df[c] = raw[c].values[idx]
    gap = np.diff(t_raw).max() if len(t_raw) > 1 else 0
    if gap > 1.5 * TS: print(f'  aviso {os.path.basename(fname)}: hueco de {gap:.1f} s entre muestras')
    return df

for f in [CSV_A, CSV_B]:
    if not os.path.exists(f):
        raise FileNotFoundError(f'Falta {f}: correr la adquisición (sección 2) antes del análisis')
dfA, dfB = load_csv(CSV_A, T_END), load_csv(CSV_B, T_END)
dfC = load_csv(CSV_C, C_T_END) if os.path.exists(CSV_C) else None
if dfC is None: print('Aviso: no hay expC.csv; se omite la sección 9.')

for name, df in [('A', dfA), ('B', dfB)] + ([('C', dfC)] if dfC is not None else []):
    print(f'Exp {name}: {len(df)} muestras, T1 {df.T1.min():.1f}–{df.T1.max():.1f} °C, '
          f'T2 {df.T2.min():.1f}–{df.T2.max():.1f} °C')

fig, ax = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
for j, (name, df) in enumerate([('A (Q1)', dfA), ('B (Q2)', dfB)]):
    ax[0, j].plot(df.time, df.T1, label='T1'); ax[0, j].plot(df.time, df.T2, label='T2')
    ax[0, j].set_title(f'Experimento {name}'); ax[0, j].set_ylabel('°C'); ax[0, j].legend()
    ax[1, j].step(df.time, df.Q1, label='Q1'); ax[1, j].step(df.time, df.Q2, label='Q2')
    ax[1, j].set_ylabel('%'); ax[1, j].set_xlabel('t [s]'); ax[1, j].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig01_datos.png'), dpi=150); plt.show()

## 4. Modelo físico 2×2 (balance de energía + sensor)

Basado en el modelo de APMonitor: cada calentador intercambia calor con el ambiente (convección + radiación) y con el
otro calentador; el sensor sigue a su disipador con un retardo de orden β (τs·D^β e + e = w). Con β = 1 es de orden
entero. Aquí **no se usa para generar datos**: es el modelo de caja gris que se ajusta a las mediciones en la sección 6.
Los valores de `PHYS` son las constantes geométricas y el punto de arranque del ajuste.

In [ ]:
PHYS = dict(Ta=23.0, U=4.0, Us=10.0, A=12e-4, As=2e-4, m=4e-3, cp=500.0, eps=0.9,
            alpha1=0.01, alpha2=0.0075, tau_s=20.0, sensor_order=1.0)

def gl_weights(a, n):
    """Pesos de Grünwald-Letnikov g_j para el orden a (g_0 = 1)."""
    return np.cumprod(np.r_[1.0, 1.0 - (a + 1.0) / np.arange(1, n)])

def phys_model(Q1, Q2, p, ts=TS, nsub=10, Ta=None):
    """Modelo físico del TCLab. Convención: y_k se lee en t_k, antes de aplicar Q_k."""
    sig = 5.67e-8
    TaC = p['Ta'] if Ta is None else Ta
    TaK = TaC + 273.15
    U, Us, A, As, mcp, eps = p['U'], p['Us'], p['A'], p['As'], p['m'] * p['cp'], p['eps']
    a1, a2, tau_s, beta = p['alpha1'], p['alpha2'], p['tau_s'], p.get('sensor_order', 1.0)
    N = len(Q1); h = ts / nsub
    TH1 = TH2 = TaK
    w = np.zeros((N, 2)); e = np.zeros((N, 2))
    g = gl_weights(beta, N); c = tau_s * ts ** (-beta); den = c * g[0] + 1.0
    for k in range(N):
        if k > 0:
            hist = g[1:k + 1] @ e[k - 1::-1]
            e[k] = (w[k] - c * hist) / den
        q1, q2 = Q1[k], Q2[k]
        for _ in range(nsub):
            conv1 = U * A * (TaK - TH1); rad1 = eps * sig * A * (TaK**4 - TH1**4)
            conv2 = U * A * (TaK - TH2); rad2 = eps * sig * A * (TaK**4 - TH2**4)
            c12 = Us * As * (TH2 - TH1); r12 = eps * sig * As * (TH2**4 - TH1**4)
            dT1 = (conv1 + rad1 + c12 + r12 + a1 * q1) / mcp
            dT2 = (conv2 + rad2 - c12 - r12 + a2 * q2) / mcp
            TH1 += h * dT1; TH2 += h * dT2
        if k + 1 < N:
            w[k + 1] = [TH1 - TaK, TH2 - TaK]
    T = TaC + e
    return T[:, 0], T[:, 1]

## 5. Modelos lineales y ajuste por mínimos cuadrados

Se trabaja en variables de desviación respecto al estado inicial. El retardo θ es continuo (interpolación entre
muestras), lo que suaviza la función de costo. El FFOPDT se simula con Grünwald-Letnikov implícito de memoria completa;
con α = 1 se reduce exactamente al FOPDT (Euler implícito), así que los dos modelos son comparables.
Las incertidumbres (±1,96σ) salen del jacobiano del ajuste.

In [ ]:
def delayed(u, theta, ts=TS):
    d = theta / ts; n = int(np.floor(d)); f = d - n; N = len(u)
    u0 = np.zeros(N); u1 = np.zeros(N)
    if n < N: u0[n:] = u[:N - n]
    if n + 1 < N: u1[n + 1:] = u[:N - n - 1]
    return (1 - f) * u0 + f * u1

def sim_fopdt(p, u, ts=TS):
    K, tau, th = p; ud = delayed(u, th); a = np.exp(-ts / tau)
    y = np.zeros(len(u))
    for k in range(1, len(u)):
        y[k] = a * y[k - 1] + K * (1 - a) * ud[k - 1]
    return y

def sim_sopdt(p, u, ts=TS):
    K, t1, t2, th = p; ud = delayed(u, th)
    a1, a2 = np.exp(-ts / t1), np.exp(-ts / t2)
    x = np.zeros(len(u)); y = np.zeros(len(u))
    for k in range(1, len(u)):
        x[k] = a1 * x[k - 1] + K * (1 - a1) * ud[k - 1]
        y[k] = a2 * y[k - 1] + (1 - a2) * x[k - 1]
    return y

def sim_ffopdt(p, u, ts=TS):
    """tau_a D^alpha y + y = K u(t - theta), GL implícito."""
    K, tau_a, al, th = p; ud = delayed(u, th); N = len(u)
    g = gl_weights(al, N); c = tau_a * ts ** (-al); den = c + 1.0
    y = np.zeros(N)
    for k in range(1, N):
        y[k] = (K * ud[k - 1] - c * (g[1:k + 1] @ y[k - 1::-1])) / den
    return y

def sim_fsopdt(p, u, ts=TS):
    """K/((tau1 s + 1)(tau_b s^beta + 1)) e^{-theta s}: disipador entero + elemento fraccionario (sensor/difusión)."""
    K, t1, tb, be, th = p; ud = delayed(u, th); N = len(u); a1 = np.exp(-ts / t1)
    g = gl_weights(be, N); c = tb * ts ** (-be); den = c + 1.0
    x = np.zeros(N); y = np.zeros(N)
    for k in range(1, N):
        x[k] = a1 * x[k - 1] + K * (1 - a1) * ud[k - 1]
        y[k] = (x[k - 1] - c * (g[1:k + 1] @ y[k - 1::-1])) / den
    return y

MODELS = {
    'FOPDT':  dict(fn=sim_fopdt,  names=['K', 'tau', 'theta'],
                   lb=[1e-3, 1, 0], ub=[5, 3000, 300]),
    'SOPDT':  dict(fn=sim_sopdt,  names=['K', 'tau1', 'tau2', 'theta'],
                   lb=[1e-3, 1, 0.5, 0], ub=[5, 3000, 1000, 300]),
    'FFOPDT': dict(fn=sim_ffopdt, names=['K', 'tau_a', 'alpha', 'theta'],
                   lb=[1e-3, 0.5, 0.3, 0], ub=[5, 1e5, 1.9, 300]),
    'FSOPDT': dict(fn=sim_fsopdt, names=['K', 'tau1', 'tau_b', 'beta', 'theta'],
                   lb=[1e-3, 1, 0.5, 0.3, 0], ub=[5, 3000, 1e4, 1.5, 300]),
}

def two_point(t, u, y, t_step):
    """FOPDT por dos puntos 35,3 % / 85,3 % (Sundaresan-Krishnaswamy)."""
    m = t >= t_step; tt, yy = t[m] - t_step, y[m]
    u_pre = u[~m][-1] if (~m).any() else 0.0
    du = u[m][-1] - u_pre; dy = yy[-30:].mean() - yy[:5].mean()
    r = (yy - yy[:5].mean()) / dy
    t1 = tt[np.argmax(r >= 0.353)]; t2 = tt[np.argmax(r >= 0.853)]
    return dict(K=dy / du, tau=0.67 * (t2 - t1), theta=max(1.3 * t1 - 0.29 * t2, 0.0))

def starts(name, g2, rng):
    K, tau, th = g2['K'], max(g2['tau'], 5), g2['theta']
    base = {'FOPDT': [K, tau, th], 'SOPDT': [K, 0.7 * tau, 0.3 * tau, 0.5 * th],
            'FFOPDT': [K, tau, 1.0, th], 'FSOPDT': [K, 0.7 * tau, 0.3 * tau, 1.0, 0.5 * th]}[name]
    out = [np.array(base, float)]
    for _ in range(N_STARTS - 1):
        p = np.array(base) * rng.uniform(0.6, 1.5, len(base))
        if name == 'FFOPDT':
            p[2] = rng.uniform(0.6, 1.5); p[1] = tau ** p[2]
        if name == 'FSOPDT':
            p[3] = rng.uniform(0.6, 1.2); p[2] = (0.3 * tau) ** p[3]
        out.append(p)
    lb, ub = np.array(MODELS[name]['lb']), np.array(MODELS[name]['ub'])
    return [np.clip(p, lb + 1e-6, ub - 1e-6) for p in out]

def fit(name, u, y, idx, g2, seed=0):
    M = MODELS[name]; fn = M['fn']; n = idx.stop
    rng = np.random.default_rng(seed)
    res = lambda p: fn(p, u[:n])[idx] - y[idx]
    best = None
    for p0 in starts(name, g2, rng):
        r = least_squares(res, p0, bounds=(M['lb'], M['ub']), x_scale='jac', max_nfev=400)
        if best is None or r.cost < best.cost: best = r
    J = best.jac; npar = len(best.x); dof = max(len(best.fun) - npar, 1)
    s2 = 2 * best.cost / dof
    try: se = np.sqrt(np.diag(s2 * np.linalg.pinv(J.T @ J)))
    except Exception: se = np.full(npar, np.nan)
    return best.x, se

def metrics(yhat, y, idx_id, idx_val, npar):
    e_id, e_val = yhat[idx_id] - y[idx_id], yhat[idx_val] - y[idx_val]
    n = len(e_id); sse = float(e_id @ e_id)
    return dict(RMSE_id=np.sqrt(sse / n), RMSE_val=np.sqrt(np.mean(e_val**2)),
                AIC=n * np.log(sse / n) + 2 * npar, BIC=n * np.log(sse / n) + npar * np.log(n))

## 6. Identificación por canal (primer escalón) y validación (escalones 55 % y 15 %)

In [ ]:
CHANNELS = [('G11', dfA, 'Q1', 'T1'), ('G21', dfA, 'Q1', 'T2'),
            ('G12', dfB, 'Q2', 'T1'), ('G22', dfB, 'Q2', 'T2')]
idx_id = slice(int(ID_WINDOW[0] / TS), int(ID_WINDOW[1] / TS))
idx_val = slice(int(VAL_WINDOW[0] / TS), int(VAL_WINDOW[1] / TS))

rows, params, preds = [], {}, {}
t0_clock = time.time()
for ch, df, qcol, tcol in CHANNELS:
    t = df.time.values; u = df[qcol].values.astype(float)
    y = df[tcol].values - df[tcol].values[:5].mean()
    g2 = two_point(t[idx_id], u[idx_id], y[idx_id], STEP_PLAN[1][0])
    preds[ch] = {'y': y, 't': t}
    yh = sim_fopdt([g2['K'], max(g2['tau'], 1), g2['theta']], u)
    rows.append(dict(canal=ch, modelo='FOPDT-2pt', npar=3, **metrics(yh, y, idx_id, idx_val, 3), **g2))
    preds[ch]['FOPDT-2pt'] = yh
    for name in MODELS:
        p, se = fit(name, u, y, idx_id, g2)
        yh = MODELS[name]['fn'](p, u); preds[ch][name] = yh
        par = dict(zip(MODELS[name]['names'], p))
        par.update({f'se_{k}': s for k, s in zip(MODELS[name]['names'], se)})
        params[(ch, name)] = par
        rows.append(dict(canal=ch, modelo=name, npar=len(p),
                         **metrics(yh, y, idx_id, idx_val, len(p)), **par))
    print(f'{ch} listo ({time.time() - t0_clock:.0f} s)')

## 7. Modelo físico 2×2 (caja gris): sensor entero contra sensor fraccionario

Se ajustan U, Us, α1, α2 y τs con los primeros escalones de ambos experimentos a la vez; T∞ es la temperatura inicial
medida. Se ajusta dos veces: con el sensor de orden entero (β = 1) y con β libre. **Esta es la comparación central de H1**:
el modelo físico captura la no linealidad (radiación, acoplamiento), así que la única diferencia entre ambos es el orden
fraccionario. El modelo con β libre es el precursor directo de la fPINN de NB02.

In [ ]:
PH_NAMES = ['U', 'Us', 'alpha1', 'alpha2', 'tau_s']

def phys_predict(theta, df, n=None, frac=False):
    p = dict(PHYS); p.update(zip(PH_NAMES, theta[:5])); p['sensor_order'] = theta[5] if frac else 1.0
    n = len(df) if n is None else n
    Ta = df[['T1', 'T2']].values[:5].mean()
    return phys_model(df.Q1.values[:n], df.Q2.values[:n], p, nsub=2, Ta=Ta)

def fit_phys(dfs, frac=False):
    def r(theta):
        out = []
        for df in dfs:
            n = idx_id.stop; T1, T2 = phys_predict(theta, df, n=n, frac=frac)
            out += [T1 - df.T1.values[:n], T2 - df.T2.values[:n]]
        return np.concatenate(out)
    lb, ub = [1, 1, 1e-3, 1e-3, 1], [30, 100, 0.05, 0.05, 150]
    x0 = [8.0, 15.0, 0.01, 0.0075, 15.0]
    if frac: lb, ub, x0 = lb + [0.3], ub + [1.5], x0 + [0.9]
    sol = least_squares(r, x0, bounds=(lb, ub), x_scale='jac', max_nfev=300)
    J = sol.jac; s2 = 2 * sol.cost / max(len(sol.fun) - len(sol.x), 1)
    se = np.sqrt(np.diag(s2 * np.linalg.pinv(J.T @ J)))
    return sol.x, se

def phys_rows(theta, se, dfs_channels, label, frac):
    out, pr = [], {}
    names = PH_NAMES + (['beta'] if frac else [])
    par = dict(zip(names, theta)); par.update({f'se_{k}': v for k, v in zip(names, se)})
    for ch, df, qcol, tcol in dfs_channels:
        T1, T2 = phys_predict(theta, df, frac=frac)
        y = df[tcol].values - df[tcol].values[:5].mean()
        yh = (T1 if tcol == 'T1' else T2) - df[tcol].values[:5].mean()
        pr[ch] = yh
        out.append(dict(canal=ch, modelo=label, npar=len(theta), **metrics(yh, y, idx_id, idx_val, len(theta)), **par))
    return out, pr, par

phys = {}
for label, frac in [('Físico entero', False), ('Físico fracc.', True)]:
    th, se = fit_phys([dfA, dfB], frac=frac)
    rr, pr, par = phys_rows(th, se, CHANNELS, label, frac)
    rows += rr; phys[label] = par
    for ch in pr: preds[ch][label] = pr[ch]
    print(label, {k: round(v, 4) for k, v in par.items() if not k.startswith('se_')})

res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, 'tabla_modelos.csv'), index=False)
cols = ['canal', 'modelo', 'npar', 'RMSE_id', 'RMSE_val', 'AIC', 'K', 'tau', 'tau1', 'tau2', 'tau_b', 'beta',
        'tau_a', 'alpha', 'theta']
print(res[[c for c in cols if c in res]].sort_values(['canal', 'AIC']).round(3).to_string(index=False))

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 8))
for a, (ch, *_ ) in zip(axs.ravel(), CHANNELS):
    P = preds[ch]
    a.axvspan(*ID_WINDOW, color='0.92', label='identificación')
    a.plot(P['t'], P['y'], 'k.', ms=1.5, label='medido')
    for name, st in [('FOPDT', '--'), ('SOPDT', '-.'), ('FSOPDT', '-'), ('Físico entero', ':'), ('Físico fracc.', '-')]:
        a.plot(P['t'], P[name], st, lw=1.3, label=name)
    a.set_title(ch); a.set_xlabel('t [s]'); a.set_ylabel('ΔT [°C]')
axs[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig02_modelos.png'), dpi=150); plt.show()

## 8. Modelos locales por punto de operación

Se ajustan FOPDT y FFOPDT en cada escalón por separado. Si K, τ (y α) cambian con el nivel, el proceso es no lineal
y un modelo global fijo no basta: es el argumento para un sustituto no lineal (fPINN) en NB02.

In [ ]:
loc = []
for ch, df, qcol, tcol in [CHANNELS[0], CHANNELS[2]]:      # G11 (control) y G12 (perturbación)
    for (ta, tb) in SEGMENTS:
        m = slice(int(ta / TS), int(tb / TS))
        k0 = int(ta / TS)
        t = df.time.values[m]
        u = df[qcol].values[m].astype(float) - df[qcol].values[k0 - 1]   # desviación respecto al nivel previo
        y = df[tcol].values[m] - df[tcol].values[k0 - 5:k0].mean()
        g2 = two_point(t, u, y, t[0])
        idx = slice(0, len(y))
        for name in ['FOPDT', 'FFOPDT']:
            p, se = fit(name, u, y, idx, g2)
            loc.append(dict(canal=ch, tramo=f'{step_profile(ta - 1):.0f}→{step_profile(ta):.0f} %', modelo=name,
                            **dict(zip(MODELS[name]['names'], p)),
                            **({'se_alpha': se[2]} if name == 'FFOPDT' else {})))
loc = pd.DataFrame(loc)
loc.to_csv(os.path.join(OUT_DIR, 'tabla_modelos_locales.csv'), index=False)
print(loc.round(3).to_string(index=False))

## 9. Veredicto H1

Criterio fijado **antes** de ver los datos reales. Una diferencia estadísticamente significativa no basta: con miles de
muestras, cualquier α ≠ 1 por error de estructura sale "significativo". Se exige además relevancia práctica.

**Comparación central (modelo físico, sensor entero vs fraccionario):**
- **Evidencia** si |β − 1| ≥ 0,05, el IC 95 % de β excluye 1 **y** el RMSE de validación mejora más de 10 %.
- **Débil** si se cumple solo la condición sobre β o solo la mejora.
- **Sin evidencia** en otro caso: el TCLab se describe bien con orden entero.

**Apoyo (modelos lineales, ventana de identificación):** ΔAIC = AIC(FSOPDT) − AIC(SOPDT) < −10 con β ≤ 0,95.
Solo cuenta β < 1 (memoria/difusión). Un β > 1 suele indicar un orden entero que falta en el modelo, no fraccionalidad:
en la prueba de control con planta entera (versión anterior del NB01), el canal cruzado G12 dio β ≈ 1,2 porque ese camino tiene tres dinámicas en serie.

In [ ]:
BETA_TOL, GAIN_TOL, DAIC_TOL = 0.05, 10.0, -10.0

def verdict_phys(tab, par):
    b, s = par['beta'], par['se_beta']
    t = tab[tab.modelo.isin(['Físico entero', 'Físico fracc.'])].pivot(index='canal', columns='modelo', values='RMSE_val')
    gain = 100 * (t['Físico entero'] - t['Físico fracc.']) / t['Físico entero']
    beta_ok = abs(b - 1) >= BETA_TOL and abs(b - 1) > 1.96 * s
    g = gain.mean()
    v = 'EVIDENCIA' if (beta_ok and g > GAIN_TOL) else ('débil' if (beta_ok or g > GAIN_TOL) else 'sin evidencia')
    return dict(beta=b, IC95=f'[{b - 1.96 * s:.3f}, {b + 1.96 * s:.3f}]', mejora_val_media_pct=g, veredicto=v), gain

def verdict_linear(tab, prm):
    out = []
    for ch, *_ in CHANNELS:
        r = tab[tab.canal == ch].set_index('modelo')
        d = r.loc['FSOPDT', 'AIC'] - r.loc['SOPDT', 'AIC']; b = prm[(ch, 'FSOPDT')]['beta']
        out.append(dict(canal=ch, beta_FSOPDT=b, dAIC_FSOPDT_SOPDT=d,
                        RMSE_val_SOPDT=r.loc['SOPDT', 'RMSE_val'], RMSE_val_FSOPDT=r.loc['FSOPDT', 'RMSE_val'],
                        apoya=bool(d < DAIC_TOL and b <= 1 - BETA_TOL)))
    return pd.DataFrame(out)

vp, gain_ch = verdict_phys(res, phys['Físico fracc.'])
vl = verdict_linear(res, params)
print('Modelo físico (comparación central):', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in vp.items()})
print('Mejora de RMSE_val por canal [%]:', gain_ch.round(1).to_dict())
print('\nModelos lineales (apoyo):'); print(vl.round(3).to_string(index=False))
pd.DataFrame([vp]).to_csv(os.path.join(OUT_DIR, 'veredicto_H1.csv'), index=False)
vl.to_csv(os.path.join(OUT_DIR, 'veredicto_H1_lineales.csv'), index=False)

g11 = res[(res.canal == 'G11') & res.modelo.isin(['FOPDT-2pt', 'FOPDT'])][['modelo', 'K', 'tau', 'theta']]
print('\nComparación con IFAC 2020 (G11):'); print(g11.round(2).to_string(index=False)); print('Referencia:', REF_IFAC)

## 10. Experimento C (APRBS): excitación y validación cruzada

**9a. ¿La señal mueve la planta lo suficiente?** Para cada nivel se mide cuánto cambia T1 durante su permanencia y se
compara con la cuantización del sensor (0,32 °C): un nivel cuenta como informativo si |ΔT1| ≥ 3 cuantos (≈ 1 °C).
También se calcula qué fracción de la varianza de T1 cae en la **banda media** ω ∈ [1/τ, 1/τs] ≈ [1/185, 1/20] rad/s (τ ≈ 185 s en este kit),
donde se distinguen el sensor y el orden fraccionario, y se compara con el experimento A (escalones).
Si muy pocos niveles son informativos, alargar `C_HOLD[0]`; si la banda media queda vacía, acortarlo.

**9b. Validación cruzada.** Los modelos de G11 y los modelos físicos se ajustaron con el **primer escalón** del
experimento A; aquí se evalúan, sin reajustar, sobre la señal APRBS. Es la prueba de generalización más exigente del
notebook y la que se reportará en el artículo junto a la validación con escalones.

In [ ]:
from scipy.signal import welch
_trap = getattr(np, 'trapezoid', None) or np.trapz

def excitation_report(df, plan, t_end, qcol='Q1', tcol='T1', quant=QUANT, band=(1/185, 1/20)):
    T = df[tcol].values; dts = []
    edges = [t for t, _ in plan[1:]] + [t_end]
    for a, b in zip(edges[:-1], edges[1:]):
        ia, ib = int(a / TS), min(int(b / TS), len(T)) - 1
        if ib - ia < 3: continue
        dts.append(T[max(ib - 3, ia):ib + 1].mean() - T[max(ia - 3, 0):ia + 1].mean())
    dts = np.abs(np.array(dts))
    def band_share(x):
        f, P = welch(x - x.mean(), fs=1 / TS, nperseg=min(1024, len(x)))
        w = 2 * np.pi * f; m = (w >= band[0]) & (w <= band[1])
        return float(_trap(P[m], f[m]) / _trap(P, f))
    return dict(niveles=len(dts), dT_mediana=float(np.median(dts)), dT_min=float(dts.min()),
                pct_informativos=100 * float(np.mean(dts >= 3 * quant)),
                banda_media_u_pct=100 * band_share(df[qcol].values.astype(float)),
                banda_media_T_pct=100 * band_share(T))

if dfC is not None:
    exc = pd.DataFrame([dict(experimento='A (escalones)', **excitation_report(dfA, STEP_PLAN, T_END)),
                        dict(experimento='C (APRBS)', **excitation_report(dfC, C_PLAN, C_T_END))])
    exc.to_csv(os.path.join(OUT_DIR, 'expC_excitacion.csv'), index=False)
    print('9a. Excitación'); print(exc.round(2).to_string(index=False))

    # 9b. validación cruzada sobre la APRBS (sin reajustar)
    u = dfC.Q1.values.astype(float); y = dfC.T1.values - dfC.T1.values[:5].mean()
    predC, vrows = {}, []
    g2 = res[(res.canal == 'G11') & (res.modelo == 'FOPDT-2pt')].iloc[0]
    predC['FOPDT-2pt'] = sim_fopdt([g2.K, max(g2.tau, 1), g2.theta], u)
    for name in MODELS:
        predC[name] = MODELS[name]['fn'](np.array([params[('G11', name)][k] for k in MODELS[name]['names']]), u)
    for label, frac in [('Físico entero', False), ('Físico fracc.', True)]:
        par = phys[label]; th = [par[k] for k in PH_NAMES] + ([par['beta']] if frac else [])
        T1h, _ = phys_predict(np.array(th), dfC, frac=frac)
        predC[label] = T1h - dfC.T1.values[:5].mean()
    for name, yh in predC.items():
        e = yh - y
        vrows.append(dict(modelo=name, RMSE_C=float(np.sqrt(np.mean(e**2))), MAE_C=float(np.mean(np.abs(e))),
                          err_max_C=float(np.max(np.abs(e)))))
    valC = pd.DataFrame(vrows).sort_values('RMSE_C')
    valC.to_csv(os.path.join(OUT_DIR, 'expC_validacion.csv'), index=False)
    print('\n9b. Validación sobre APRBS (G11, modelos del primer escalón de A)'); print(valC.round(3).to_string(index=False))

    fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw=dict(height_ratios=[1, 2]))
    ax[0].step(dfC.time, dfC.Q1, where='post'); ax[0].set_ylabel('Q1 [%]'); ax[0].set_title('Experimento C (APRBS)')
    ax[1].plot(dfC.time, y, 'k.', ms=1.5, label='medido')
    for name, st in [('FOPDT', '--'), ('SOPDT', '-.'), ('FSOPDT', '-'), ('Físico entero', ':'), ('Físico fracc.', '-')]:
        ax[1].plot(dfC.time, predC[name], st, lw=1.2, label=name)
    ax[1].set_ylabel('ΔT1 [°C]'); ax[1].set_xlabel('t [s]'); ax[1].legend(fontsize=8, ncol=3)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig03_expC.png'), dpi=150); plt.show()

## 11. Notas para la discusión

- El retardo aparente del TCLab proviene de la dinámica disipador→sensor, no de transporte. Por eso el competidor que
  pedirá un revisor es el SOPDT y el modelo físico de dos estados, no el FOPDT.
- El FFOPDT de un solo elemento se incluye porque es el modelo fraccionario estándar en la literatura de procesos,
  pero no puede representar dos dinámicas en serie: si pierde frente al SOPDT, eso no descarta la fraccionalidad.
  La comparación justa es SOPDT contra FSOPDT y modelo físico entero contra fraccionario.
- La validación de los modelos lineales está dominada por la no linealidad (K y τ cambian entre tramos, sección 8);
  por eso el veredicto central usa el modelo físico, que sí la captura.
- El canal G12 (Q2→T1) es el camino de la perturbación del lazo SISO del artículo. Es de orden mayor (calentador 2 →
  calentador 1 → sensor), por eso los modelos de un elemento le asignan órdenes > 1 aunque la planta sea entera.
- Si el SOPDT sale con θ ≈ 0 en G11, se confirma que el "tiempo muerto" del FOPDT es aparente.
- Salidas en `NB01_out/`: datos CSV, tablas de modelos, veredictos, metadatos del kit (`metadatos_kit.json`) y figuras.
- Experimento C: semilla `C_SEED` fija; el plan se guarda en `plan_expC.csv` para reproducir la señal exacta.

## 12. Empaquetar resultados para compartir

Crea `NB01_resultados_<fecha>.zip` (junto al notebook) con todo `NB01_out/`: datos crudos de cada experimento
(`expA.csv`, `expB.csv`, `expC.csv` y cualquier `.part.csv` de una corrida cortada), plan de la APRBS, tablas de
parámetros identificados, veredictos, metadatos del kit y figuras. Ese .zip es el archivo para subir al proyecto.

In [ ]:
import shutil
res_par = pd.DataFrame([dict(canal=ch, modelo=m, **p) for (ch, m), p in params.items()])
res_par.to_csv(os.path.join(OUT_DIR, 'parametros_lineales.csv'), index=False)
pd.DataFrame([dict(modelo=k, **v) for k, v in phys.items()]).to_csv(os.path.join(OUT_DIR, 'parametros_fisicos.csv'), index=False)
zip_name = shutil.make_archive(f'NB01_resultados_{datetime.date.today():%Y%m%d}', 'zip', root_dir='.', base_dir=OUT_DIR)
print('Archivo para subir:', os.path.abspath(zip_name))
for f in sorted(os.listdir(OUT_DIR)):
    print(f'  {f:35s} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1024:8.1f} KB')